# 003 - Local Indexing

Written by Jean-Baptiste Jacob

Last updated: 19/02/2026

Perform local indexing of orientation and unit cell pixel-by-pixel and map the results on a 2D grid (Pixelmap). This notebook is for testing and tuning the indexing parameters. Use fp3b_Local_indexing_batch.ipynb to send jobs to slurm. 

### Load packages

In [ ]:
import sys

# python environment stuff
IMAGED11_PATH = '/home/esrf/jean1994b/ImageD11_jbjacob'  # None means do not use git, otherwise enter the name of the folder to use for the git checkout "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = 'ImageD11'  # the name of the git checkout folder within path. None means guess

if IMAGED11_PATH is not None:
    if '/data/id11/nanoscope' not in sys.path:
        sys.path.append('/data/id11/nanoscope')
    import install_ImageD11_from_git
    PYTHONPATH = install_ImageD11_from_git.setup_ImageD11_from_git(IMAGED11_PATH,CHECKOUT_PATH)
else:
    import site
    PYTHONPATH = site.getsitepackages()[0]
    print(PYTHONPATH)

In [ ]:
# general modules
import os, glob, time
import h5py
import matplotlib.pyplot as pl
import numpy as np
from tqdm import tqdm

# ImageD11 https://github.com/FABLE-3DXRD/ImageD11
import ImageD11.sinograms.dataset
import ImageD11.columnfile
import ImageD11.parameters
import ImageD11.indexing
import ImageD11.grain
import ImageD11.sym_u
import ImageD11.friedel_pairs as fp

# point-fit 3dxrd module available at https://github.com/jbjacob94/pf_3dxrd.   
from pf3dxrd.pf3dxrd import utils, pixelmap, crystal_structure, peak_mapping, local_indexing, refine_ubi

%load_ext autoreload
%autoreload 2
%matplotlib ipympl

### Load data
- filtered peakfile (saved from notebook 002_Phase_Labelling)
- 2D map (Pixelmap)$
- dataset object

In [ ]:
# loading function. we need dataset (ds), pixelmap (xmap), peakfile (cf) and crystal structure (cs)
def load_data(dsfile, phase):
    # dataset
    ds = ImageD11.sinograms.dataset.load(dsfile)
    print(ds)
    
    # paths for xmap and peaks
    xmapfile = ds.dsfile.replace('dataset.h5','xmap.h5')
    col2dfile = ds.col2dfile.replace('.h5','_paired.h5')

    # load
    xmap = pixelmap.load_from_hdf5(xmapfile)
    print(xmap)
    # crystal structure we want to index
    cs = xmap.phases.get(phase)
    pid = cs.phase_id
    
    # load cf  + keep only peaks from the phase we want to index
    cf = ImageD11.columnfile.columnfile(col2dfile)
    cf.parameters.loadparameters(ds.parfile)
    fp.update_geometry_fpairs(cf,ds)
    cf.filter(cf.phase_ids==pid)
    utils.get_colf_size(cf)
    print(f'N peaks: {cf.nrows}')
    
    # add pixel labeling to cf + sort by pixel index
    if 'xyi' not in cf.titles:
        peak_mapping.add_pixel_labels(cf, ds = ImageD11.sinograms.dataset.load(dsfile))
    if not cf.sortedby == 'xyi':
        print('sorting peakfile by "xyi"...')
        cf.sortby('xyi')
    
    return xmap, cf, cs, ds


In [ ]:
# datase path and phase to index
dsfile = 'MgO_3_0p1M_12um_0003/MgO_3_0p1M_12um_0003_dataset.h5'
phase = 'MgO'

In [ ]:
# load data
xmap, cf, cs, ds = load_data(dsfile, phase)

Note that crystal structures have been saved has phase attributes when saving pixelmap in a HDF5 file, so they don't need to be imported again from cif files. 

### Local indexing

In [ ]:
# initialize an Option instance which will be passed to the indexing function
OPTS = local_indexing.Options()
print(OPTS)

#### options definition & default values

- `hkltol1`        = 0.05      # hkl tolerance parameter for indexing
- `hkltol2`        = 0.03      # hkl tolerance parameter for refinement
- `useIntensity`   = False     # if True, include intensity score for matching best ubi (refinement stage)
- `minpks`         = 10        # minimum number of g-vectors to consider a ubi as a possible match
- `minpks_prop`    = 0.1       # min fraction of g-vectors over pixel to consider a ubi match.   The value passed to the indexer is calculated as:  
`max(minpks, len(gvecs) * minpks_prop)`  
- `maxpks`         = 5000      # cutoff value for peaks number in 1st-round indexing
- `frac_strong`    = 0.9       # cumulative intensity fraction of strong peaks selection for indexing (1st stage) 
- `nrings`         = 10        # max number of hkl rings to search
- `max_mult`       = 12        # max multiplicity of hkl rings to search
- `ds_tol`         = 0.005     # ds tolerance in ImageD11.indexer
- `cosine_tol`     = np.cos(np.radians(90 - 0.5))    # cosine tolerance in ImageD11.indexer
- `px_kernel_size` = 3         # kernel size around pixel (1 = single pixel)
- `chunksize`      = 20        # chunk size for ProcessPoolExecutor
- `symmetry`       = 'cubic'   # crystal symmetry. must be one of ['cubic', 'hexagonal', 'trigonal', 'rhombohedralP', 'trigonalP', 'tetragonal', 'orthorhombic', 'monoclinic_c', 'monoclinic_a', 'monoclinic_b', 'triclinic']
- `symmetrize_ubi` = True      # If True, return the symmetry-reduced ubi for the given symmetry. **set to True unless custom flipmats have been specified**
 
##### ---- Derived or system-dependent parameters ----
- `unitcell` = None    # crystallographic unit cell of the phase being indexed. This is read from the crystal structure (cs) object
- `sym`      # ImageD11.sym_u object, initialized using OPTS.symmetry
- `flipmats` = None  # list of flip matrices. **expert mode. use only if you know what you're doing!**
- `ncpu`   # number of cpus when running the script locally on the jupyter session (usecluster=False). use all but one CPU by default
 
##### ---- slurm / cluster options ----
partition, memory allocated, time, cpus


Update symmetry and unit cell for the phase to index, and any other parameter if needed

In [ ]:
OPTS.setsymmetry('hexagonal')
OPTS.unitcell = cs.to_ImageD11_unitcell()
OPTS.hkltol1 = 0.1
OPTS.hkltol2 = 0.04
OPTS.maxpks = 1500
OPTS.max_mult = 24
OPTS.ds_tol = 0.005
OPTS.px_kernel_size = 1
OPTS.symmetrize_ubi = True
OPTS.nrings = 10
OPTS.chunksize = 10
OPTS.useIntensity = True
OPTS.slurm_cpus = 48

In [ ]:
OPTS.flipmats = None 

### Note on flipmats

For some trigonal and tetragonal phases, the lattice symmetry is higher than the point group symmetry. For instance, quartz has a hexagonal lattice with 6-fold symmetry, but the crystal structure is trigonal (3-fold symmetry). This means there are two possible ways to orient the trigonal cell inside that hexagonal "box," related by a 60° rotation about the c-axis (equivalently 180°, since these are equivalent under hexagonal symmetry).

This causes a problem for indexing, since the indexing algorithm only sees the lattice, not the variation in peak intensities associated with the 6-fold rotation. Therefore, quartz must first be indexed in the hexagonal symmetry (`OPTS.setsymmetry('hexagonal')`); then, during refinement, all possible trigonal flips are tested to identify the best-matching one. These flips are encoded by the `flipmats` parameter, a list of 3x3 matrices.

### Example: quartz

Starting from an indexed hexagonal orientation, there are two possible trigonal orientations: the one in the hexagonal fundamental zone, and the one rotated 180° about the c-axis. These correspond to two flip matrices:

$$\begin{pmatrix}1 & 0 & 0\\0 & 1 & 0\\0 & 0 & 1\end{pmatrix} \quad \text{and} \quad \begin{pmatrix}-1 & 0 & 0\\0 & -1 & 0\\0 & 0 & 1\end{pmatrix}$$

Add these to the `flipmats` list. During refinement, for each UBI candidate found in the first indexing stage, the list of UBI matrices to score is expanded according to all possible flips in `flipmats`:

```python
for i, ubi in enumerate(ubi_uniqs):
    for j, group_rot in enumerate(OPTS.flipmats):
        ubi_rot = group_rot.dot(ubi_uniq)
        ubis.append(ubi_rot)
```

Each UBI in this expanded list is then scored. For accurate scoring in the case of quartz, `OPTS.useIntensity` must be set to `True` (otherwise the two flips are equivalent), and `OPTS.symmetrize_ubi` must be set to `False` — otherwise, in the final step after refinement, the algorithm will map the trigonal orientation back to the hexagonal fundamental zone, losing the extra information gained from intensity scoring.

In [ ]:
print(OPTS)

#### Prepare g-vectors for indexing
create a separate temporay file containing the g-vectors to index and save it. Then, multiple processes will be able to access these data at the same time. Optionally, it is possible to filter the g-vectors to index (e.g. filter by intensity to index only the strongest peaks). 


In [ ]:
# save g-vectors to index in a temporary file
# g-vectors to index. take only useful columns with g-vector coordinates, peak intensities and pixel coordinates in the sample
to_index = ImageD11.columnfile.colfile_from_dict( { name: cf.getcolumn(name) for name in 'gx gy gz xyi norm_intensity'.split() } )

# sort by pixel index
to_index.sortby('xyi')

Select pixel indices `xyi_uniqs` to index. Using a small subset makes it much faster to index for fine-tuning input parameters. 

In [ ]:
#list of pixels to index
# FULL MAP
xyi_uniqs = xmap.xyi[xmap.phase_ids == cs.phase_id]

# RECTANGULAR SUBSET: uncomment these lines if you want to index only a subset of the map
sel = np.all([np.abs(xmap.xi-204) < 25, np.abs(xmap.yi - 250) < 25, xmap.phase_ids == cs.phase_id], axis=0)
xyi_uniqs = xmap.xyi[sel]

print(f'Number of pixels to process: {len(xyi_uniqs)}')

**General recommendations for parameters tuning:**  

**for starting**: use large hkltol thresholds (e.g. hkltol1 = 0.1, hkltol2 = 0.08), single pixels (px_kernel_size=1), no intensities (useIntensity=False), and default minpks, maxpks, frac_strong, nrings.

max_mult is adjusted depending on crystal symmetry: 24 may be ok for cubic but too high for a lower symmetry for instance. 

If execution time is too long (should take 50-500 ms per pixel, 1s max): reduce maxpks, then nrings. affects only 1st-stage indexing, which is the slowest by far, not refinement 

**fine-tuning**: if initial parameters return noisy / incomplete indexing
- increase px_kernel_size: gives more local context, which can help to improve indexing
- increase frac_strong /maxpks/hkltol1: increase the number of g-vectors for 1st stage indexing. slower but can help to find the correct ubi
- increase nrings : tries more hkl rings combination during indexing -> more ubi candidates to score 
- set useIntensity=True: add more information during refinement

**once the map looks correct**
- iteratively reduce hkltol2 until completeness starts degrading too much.
- Plot the map of average `drlv2` after indexing. Check if there is some radial increase in `drlv2` /radial decrease in `nindx` (typical for large samples >2-3mm). If so, try to relax a bit hkltol_2 to retain more g-vectors from domains far away from the rotation center. There is a tradeoff here between precision (using more stringent hkltol to keep only the best matching g-vectors) and map consistency across a large radial domain

#### Test local indexing on one pixel
make sure indexing works correctly. Computation time should be in the range of 100ms to 1s. If it is much longer, try reduce maxpks / frac_strong. If nothing changes, this may be caused by inadapted format of the xyi array in the peakfile (should be native int type rather than numpy.int).

In [ ]:
# Test on one pixel
px = xyi_uniqs[324]   # pixel to index
args = (px, OPTS)  # wrap arguments to pass to indexing function

In [ ]:
# create a context for indexing: loads to_index, update pars, etc. and closes it when finished
with local_indexing.indexing_context(ds.parfile, to_index, cs):
    t0 = time.perf_counter()
    res = local_indexing.pixel_ubi_fit(args)
    t = time.perf_counter() - t0
    print(f'results: {res}')
    print(f'time: {100*t:.4f} ms')

output structure:  tuple `(px_index, UBI, nindx, drlv2_mean, Iindx, indx_completeness, Icorrel)`:
- `px_index` is the 'xyi' index (`xyi = 10000.yi+xi`) corresponding to the indexed pixel
- `UBI` is the best match (refined) for the unit cell matrix
- `nindx` is the number of g-vectors used to refine the unit cell matrix
- `drlv2_mean` is the average drlv2 value for all g-vectors retained for fitting.
- `Iindx` is the summed intensity of indexed peaks
- `indx_completeness` is the fraction of indexed intensity over total intensity $\frac{I_{indx}}{I_{tot}}$
- `Icorrel` is the Pearson's correlation coefficient between intensity of the indexed hkls and predicted intensity from the crystal structure (computed using Dans_diffraction). set to nan if useIntensity is False  


#### Run local indexing in parallel
Now everything is set up, we can run indexing in parallel. Results are stored in a temporary dictionnary. Then we will unpack these results and update the pixelmap. 

In [ ]:
# define argument list
argslist = [(px, OPTS) for px in xyi_uniqs]

# run indexing
results = local_indexing.run_indexing_parallel(argslist, to_index, OPTS, ds.parfile, cs)

#### Result inspection: plot indexing stats
histograms of number of peaks indexed per pixel, average drlv2, completeness and intensity correlation (if computed) to assess indexing quality;

In [ ]:
stats, fig = refine_ubi.compute_refinement_stats(results)

#### Extract results and add them to pixelmap
If indexing looks good, export results to xmap: unpack results dictionnary and updates xmap columns (or create new columns if they don't exist)

By default, `update_xmap()` resets all pixels of the selected phase (`overwrite = True`), to make sure all pixels with assigned UBI have been indexed with the same parameters. 

You can also choose not to touch previously indexed pixels, and write only data where nothing has been indexed. 
Useful for testing: index to neighbour blocks with different option and compare the difference on the map

In any case, pixels belonging to other phases will not be touched. 

In [ ]:
local_indexing.update_xmap(xmap, xyi_uniqs, results, phase, overwrite = True)

In [ ]:
# optional: you can decide to filter out some pixels that are indexed poorly -> reset phase to notIndexed.
# USE WITH CAUTION: you will not be able to reindex this pixel later if the phase index is reset. 
#to_keep = (xmap.drlv2 < 4e-4) & (xmap.nindx > 15) & (xmap.indx_completeness > 0.2) & (xmap.Icorrel > 0.2)
#xmap.phase_ids[~to_keep] = -1

In [ ]:
# check new columns have been added to xmap
print(xmap)

### Plotting
plots for orientation and indexing statistics (drlv2, etc.)

In [ ]:
save=False
xmap.plot_ipf_orientation(phase=phase, datacolname='U', ipf_directions='xyz', save=save, hide_cbar=False, out=False)

In [ ]:
save=False
var_to_plot = ['nindx','drlv2', 'indx_completeness', 'Npks', 'intensity', 'Icorrel']
kw = {'cmap':'viridis'}

for var in var_to_plot:
    if var in xmap.titles():
        xmap.plot(var, phase=phase, autoscale=True, hist_tails_cut=[1,99], save=save)
        

### Save indexed map and options
once you are satisfied with indexing outputs, save the parameters in a json file. If the map has been fully indexed, you can save it as well. Nothing has been added to the peakfile, so no need to save cf.

Re-do the process for every phase to index.

In [ ]:
# indexing params
OPTS.save(f"indexing_pars_{phase}.json")

In [ ]:
# xmap saving: optional
xmap.save_to_hdf5()

In [ ]:
# also optional (if the map has been fully indexed): convert xmap to ImageD11 Tensormap and save. 
tmap = xmap.to_tensor_map()
tmap.to_h5(xmap.h5name.replace('xmap.h5','tmap.h5'))

### Local output investigation at pixel level
Run indexing on selected pixels and investigate indexed g-vectors + indexing metrics. Useful to investigate noisy pixels indexing or better tune indexing parameters

Typical issues with indexing:
- too few / bad g-vectors on noisy pixels -> try to increase kernel size to get more g-vectors from neighbors 
- maxpks/frac_strong or nrings too low: miss the "correct" orientation -> increase slightly these parameters until the correct orientation is found

In [ ]:
# select list of pixels to investigate. xyi index value: xyi = 1000.yi + xi  (e.g. (xi,yi) = (34,23) -> xyi = 230034
#pxlist = [3080079, 3190038, 3250032, 3370072, 2880089]

# or take a random list from xyi_uniqs
pxlist = np.random.choice(xyi_uniqs, size=15)

# plot them on the map to see where they are
kw = {'cmap':'viridis'}
fig = xmap.plot_ipf_orientation(phase=phase, datacolname='U', ipf_directions=[(0,0,1)], save=False, hide_cbar=False, out=True)
fig2 = xmap.plot('Npks', out=True, autoscale=True)

for px in pxlist:
    fig.axes[0].scatter(px%10000,px//10000, s=30, marker='*')
    fig2.axes[0].scatter(px%10000,px//10000, s=30, marker='*')
fig.axes[0].set_axis_on()
fig2.axes[0].set_axis_on()
  

In [ ]:
#plot pole figures of g-vectors (aligned with crystallographic c-axis pointing upward)-> check if they correspond to a unique grain
import orix.quaternion as oq
import orix.vector as ovec

def get_gvecs(cf, px, kernel_size=1):
    # for pixels. px: index xyi = 1000*yi+xi
    s = peak_mapping.pks_from_px(cf.xyi.astype(int), px, kernel_size=kernel_size)
    cut = local_indexing._strong_peaks(cf.norm_intensity[s], frac=.99, min_peaks=3, max_peaks=350000)
    s = s[cut]

    gv = np.array( (cf.gx[s],cf.gy[s],cf.gz[s])).T.copy()
    ints = cf.norm_intensity[s]
    return gv, ints, s

def plot_gvectors_polefig(cf, px, gv, ints, mask=None, rotate=False, sf=1, **subplot_kw):
    fig = pl.figure(figsize=(5,5), layout='constrained')
    fig.suptitle(f'g-vectors scatter plot', fontsize=14, fontweight="bold")

    ax = fig.add_subplot(111, **subplot_kw)
    ax.set_axis_off()
    U = xmap.U[xmap.xyi == px]
    
    if rotate:
        R = oq.Rotation.from_matrix(U)  # realign g-vectors to have c-axis pointing upward
    else:
        R = oq.Rotation.from_matrix(np.eye(3,3))
        
    if mask is None:
        mask = np.full(len(gv),True)
    V= ovec.Vector3d(gv[mask]).rotate(R.axis, -R.angle)
    ax.scatter(V, c=pow(ints[mask],1/3), s=sf*pow(ints[mask],1/3))
    ax.set_title(f'x={px%10000:d}, y={px//10000:d}')


In [ ]:
# index each pixel in debug mode, to see how ubi scores look like at each stage
results = []
with local_indexing.indexing_context(ds.parfile, to_index, cs):
    for px in pxlist:
        args = (px, OPTS)
        print(f'px : {px}')
        res = local_indexing.pixel_ubi_fit(args, loginfo=False)  # set loginfo to True for verbose mode 
        results.append(res)
        print(f'results: {res}')
        print(f'\n========================\n')

In [ ]:
# plot polefigures of indexed g-vectors

DRLV2 = []
MASKS = []
subplot_kw ={'projection':'stereographic','hemisphere':'upper'}
OPTS.maxpks = 1000
OPTS.hkltol2 = 0.08
OPTS.px_kernel_size=3

for res in results:
    px = res[0]
    ubi = res[1]

    gv, ints, s = get_gvecs(cf, px, kernel_size=OPTS.px_kernel_size)

    hkl_int, drlv2, mask = refine_ubi._compute_hkl_residuals(ubi, gv, hkl_tol=OPTS.hkltol2)
    print(f'nindx / ntot: {mask.sum()}, {len(mask)}')
    print(f'Iindx / Itot: {ints[mask].sum()}, {ints.sum()}')
    DRLV2.append(drlv2)
    MASKS.append(mask)

    ub = np.linalg.inv(ubi)
    g_pred = (ub @ hkl_int[mask].T).T
    g_meas = gv[mask]
    residual_gv = np.linalg.norm(g_pred - g_meas, axis=1)
    print(f'residuals q5,50,95:{np.percentile(residual_gv,(5,50,95))}\n------------------------------------') 

    plot_gvectors_polefig(cf, px, gv,ints, mask=mask, rotate=False, sf=1, **subplot_kw)

In [ ]:
# histogram of drlv2 for indexed peaks
for drlv2, m in zip(DRLV2,MASKS):
    pl.figure(figsize=(15,3))
    pl.hist(drlv2, bins=np.linspace(0,np.percentile(drlv2[m],99.9),100));
    pl.hist(drlv2[m], bins=np.linspace(0,np.percentile(drlv2[m],99.9),100),alpha=.5);